TCN model

importing 

In [28]:
%pip install torch --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import numpy as np
import pandas as pd
import json
from typing import Tuple, Dict, List
import os
import pickle
import torch
import torch.nn as nn
import torch.optim as optim

loading and inspecting the data

In [2]:
df = pd.read_csv('traffic_features org.csv')

print(f"Total rows: {len(df)}")
print(f"Total columns: {len(df.columns)}")
print(f"\nColumn names and types:")
print(df.dtypes)
print(f"\nFirst 5 rows:")
print(df.head())
print(f"\nMissing values:")
print(df.isnull().sum())
print(f"\nCongestion class distribution:")
print(df['target_congestion_level'].value_counts().sort_index())

Total rows: 1464
Total columns: 16

Column names and types:
timestamp                      str
location_id                    str
vehicle_count              float64
avg_speed                      str
weather                        str
day_of_week                    str
is_holiday                   int64
event                          str
sensor_status                  str
road_condition                 str
target_congestion_level        str
school_in_session            int64
lag1                       float64
roll3                      float64
hour                       float64
weekday                    float64
dtype: object

First 5 rows:
             timestamp location_id  vehicle_count           avg_speed weather  \
0  2023-01-02 00:30:00         ???           17.0   42.34024454353148   Rainy   
1  2023-01-05 02:15:00         ???           23.0   58.07196643637885   Rainy   
2  2023-01-06 14:15:00         ???           27.0   44.24393937861012   Clear   
3  2023-01-14 06:15:00     

parsing timestamp

In [3]:
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)

print(f"\nTimestamp range: {df['timestamp'].min()} to {df['timestamp'].max()}")
initial_count = len(df)


Timestamp range: 2023-01-01 03:00:00 to 2023-01-16 17:45:00


handle missing values.

In [4]:

# Convert avg_speed: '???' to NaN
df.loc[df['avg_speed'] == '???', 'avg_speed'] = np.nan
df['avg_speed'] = pd.to_numeric(df['avg_speed'], errors='coerce')

# Fill lag1 with roll3 where missing
df['lag1'] = df['lag1'].fillna(df['roll3'])

# Fill remaining lag1 with group median (hour + weekday)
for hour in sorted(df['hour'].unique()):
    for weekday in sorted(df['weekday'].unique()):
        mask = (df['hour'] == hour) & (df['weekday'] == weekday)
        group_median = df.loc[mask & df['roll3'].notna(), 'vehicle_count'].median()
        
        if pd.notna(group_median):
            df.loc[mask & df['lag1'].isna(), 'lag1'] = group_median

# Fill avg_speed with hour median
for hour in sorted(df['hour'].unique()):
    mask = df['hour'] == hour
    hour_median = df.loc[mask & df['avg_speed'].notna(), 'avg_speed'].median()
    if pd.notna(hour_median):
        df.loc[mask & df['avg_speed'].isna(), 'avg_speed'] = hour_median

# Drop rows with NaN in critical columns
critical_cols = ['vehicle_count', 'hour', 'weekday', 'target_congestion_level']
df = df.dropna(subset=critical_cols).reset_index(drop=True)

# Now fill remaining NaN values in other columns
# For location_id, weather, road_condition, fill with mode or Unknown
df['location_id'] = df['location_id'].fillna(df['location_id'].mode()[0] if len(df['location_id'].mode()) > 0 else 0)
df['weather'] = df['weather'].fillna('Unknown')
df['road_condition'] = df['road_condition'].fillna('Unknown')

# For roll3 (numeric), fill with median
df['roll3'] = df['roll3'].fillna(df['roll3'].median())

# Drop any remaining rows with NaN
df = df.dropna().reset_index(drop=True)

print(f"Rows after cleaning: {len(df)} (removed {initial_count - len(df)} rows)")
print(f"Missing values after cleaning:")
print(df.isnull().sum())


Rows after cleaning: 1083 (removed 381 rows)
Missing values after cleaning:
timestamp                  0
location_id                0
vehicle_count              0
avg_speed                  0
weather                    0
day_of_week                0
is_holiday                 0
event                      0
sensor_status              0
road_condition             0
target_congestion_level    0
school_in_session          0
lag1                       0
roll3                      0
hour                       0
weekday                    0
dtype: int64


feature engineering

In [5]:
# TEMPORAL FEATURES: Cyclic encoding (preserves continuity across boundaries)
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

df['weekday_sin'] = np.sin(2 * np.pi * df['weekday'] / 7)
df['weekday_cos'] = np.cos(2 * np.pi * df['weekday'] / 7)

# CATEGORICAL ENCODING: One-hot for weather
weather_dummies = pd.get_dummies(df['weather'].fillna('Unknown'), prefix='weather')
df = pd.concat([df, weather_dummies], axis=1)

# CATEGORICAL ENCODING: One-hot for road_condition
road_dummies = pd.get_dummies(df['road_condition'].fillna('Unknown'), prefix='road_cond')
df = pd.concat([df, road_dummies], axis=1)

# SENSOR STATUS: Binary (OK vs NotOK)
df['sensor_ok'] = (df['sensor_status'] == 'OK').astype(float)

# EVENT PRESENCE: Binary flag
df['event_present'] = (~df['event'].isna() & (df['event'] != '')).astype(float)

# Binary flags: is_holiday, school_in_session (already in data)
# Convert to float if needed
df['is_holiday'] = df['is_holiday'].astype(float)
df['school_in_session'] = df['school_in_session'].astype(float)

print(f"Encoded features:")
print(f"  Temporal (cyclic): hour_sin, hour_cos, weekday_sin, weekday_cos")
print(f"  Categorical weather: {[col for col in df.columns if col.startswith('weather_')]}")
print(f"  Categorical road: {[col for col in df.columns if col.startswith('road_cond_')]}")
print(f"  Binary: sensor_ok, event_present, is_holiday, school_in_session")
print(f"  Auto-regressive: lag1, roll3")
print(f"  Target/endogenous: vehicle_count, avg_speed")


Encoded features:
  Temporal (cyclic): hour_sin, hour_cos, weekday_sin, weekday_cos
  Categorical weather: ['weather_???', 'weather_Clear', 'weather_Foggy', 'weather_Rainy', 'weather_Unknown']
  Categorical road: ['road_cond_???', 'road_cond_Good', 'road_cond_Moderate', 'road_cond_Poor', 'road_cond_Unknown']
  Binary: sensor_ok, event_present, is_holiday, school_in_session
  Auto-regressive: lag1, roll3
  Target/endogenous: vehicle_count, avg_speed


Rolling origin split

In [6]:

total_rows = len(df)
train_end = int(0.61 * total_rows)      # 61%
val_end = int(0.76 * total_rows)        # 61% + 15%
# test_end = total_rows                  # remaining 24%

print(f"Total rows: {total_rows}")
print(f"Train: 0 to {train_end} ({100*train_end/total_rows:.1f}%)")
print(f"Val:   {train_end} to {val_end} ({100*(val_end-train_end)/total_rows:.1f}%)")
print(f"Test:  {val_end} to {total_rows} ({100*(total_rows-val_end)/total_rows:.1f}%)")

df_train = df.iloc[:train_end].reset_index(drop=True)
df_val = df.iloc[train_end:val_end].reset_index(drop=True)
df_test = df.iloc[val_end:].reset_index(drop=True)

print(f"\nSplit sizes:")
print(f"  Train: {len(df_train)} rows")
print(f"  Val:   {len(df_val)} rows")
print(f"  Test:  {len(df_test)} rows")

Total rows: 1083
Train: 0 to 660 (60.9%)
Val:   660 to 823 (15.1%)
Test:  823 to 1083 (24.0%)

Split sizes:
  Train: 660 rows
  Val:   163 rows
  Test:  260 rows


defining input features for TCN

In [7]:
# Temporal features
temporal_features = ['hour_sin', 'hour_cos', 'weekday_sin', 'weekday_cos']

# Auto-regressive features (lags)
lag_features = ['lag1', 'roll3']

# Exogenous features
weather_features = [col for col in df.columns if col.startswith('weather_')]
road_features = [col for col in df.columns if col.startswith('road_cond_')]
exogenous_features = (
    weather_features + road_features + 
    ['sensor_ok', 'event_present', 'is_holiday', 'school_in_session']
)

# Input to both Phase 1 and Phase 2
tcn_input_features = (
    temporal_features + lag_features + exogenous_features + ['vehicle_count', 'avg_speed']
)

print(f"TCN input features ({len(tcn_input_features)}):")
for i, feat in enumerate(tcn_input_features):
    print(f"  {i}: {feat}")

# Target features
regression_target = 'vehicle_count'
classification_target = 'target_congestion_level'

print(f"\nRegression target: {regression_target}")
print(f"Classification target: {classification_target}")



TCN input features (22):
  0: hour_sin
  1: hour_cos
  2: weekday_sin
  3: weekday_cos
  4: lag1
  5: roll3
  6: weather_???
  7: weather_Clear
  8: weather_Foggy
  9: weather_Rainy
  10: weather_Unknown
  11: road_cond_???
  12: road_cond_Good
  13: road_cond_Moderate
  14: road_cond_Poor
  15: road_cond_Unknown
  16: sensor_ok
  17: event_present
  18: is_holiday
  19: school_in_session
  20: vehicle_count
  21: avg_speed

Regression target: vehicle_count
Classification target: target_congestion_level


standarding

In [8]:
class StandardScaler:
    """Simple StandardScaler implementation"""
    def __init__(self):
        self.mean = None
        self.std = None
    
    def fit(self, X):
        """Fit on training data"""
        X = X.astype(float)  # Ensure numeric dtype
        self.mean = np.mean(X, axis=0)
        self.std = np.std(X, axis=0)
        # Avoid division by zero
        self.std[self.std == 0] = 1.0
    
    def transform(self, X):
        """Transform data using fitted statistics"""
        X = X.astype(float)  # Ensure numeric dtype
        return (X - self.mean) / self.std
    
    def fit_transform(self, X):
        """Fit and transform in one step"""
        self.fit(X)
        return self.transform(X)

# Initialize and fit scaler on training data ONLY
scaler = StandardScaler()

X_train = df_train[tcn_input_features].values
scaler.fit(X_train)

# Transform all splits using training statistics
X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(df_val[tcn_input_features].values)
X_test_scaled = scaler.transform(df_test[tcn_input_features].values)

# Get targets
y_train_volume = df_train[regression_target].values
y_val_volume = df_val[regression_target].values
y_test_volume = df_test[regression_target].values

y_train_congestion = pd.to_numeric(df_train[classification_target], errors='coerce').fillna(0).astype(int)
y_val_congestion = pd.to_numeric(df_val[classification_target], errors='coerce').fillna(0).astype(int)
y_test_congestion = pd.to_numeric(df_test[classification_target], errors='coerce').fillna(0).astype(int)

print(f"Training features shape: {X_train_scaled.shape}")
print(f"Training targets - volume: {y_train_volume.shape}, congestion: {y_train_congestion.shape}")
print(f"\nScaler statistics:")
print(f"  Mean: {scaler.mean[:5]}... (showing first 5)")
print(f"  Std:  {scaler.std[:5]}... (showing first 5)")

Training features shape: (660, 22)
Training targets - volume: (660,), congestion: (660,)

Scaler statistics:
  Mean: [1.76973209e-02 1.34612288e-03 2.41353206e-02 1.96641564e-01
 1.99393939e+01]... (showing first 5)
  Std:  [0.70645956 0.70753083 0.68775007 0.69839059 4.71571772]... (showing first 5)


constructing 24 hour sliding window

In [9]:
def create_tcn_windows(X_scaled, y_volume, y_congestion, window_size=24):
    """
    Create sliding windows of 24 hours for TCN input.
    
    Args:
        X_scaled: (N, num_features) scaled input features
        y_volume: (N,) vehicle counts
        y_congestion: (N,) congestion labels
        window_size: 24 hours
    
    Returns:
        X_windows: (num_windows, 24, num_features)
        y_vol_windows: (num_windows,) target volume
        y_cong_windows: (num_windows,) target congestion
    """
    
    X_windows = []
    y_vol_windows = []
    y_cong_windows = []
    
    # Create sliding windows
    for i in range(len(X_scaled) - window_size):
        window = X_scaled[i:i+window_size]  # 24 timesteps
        
        # Target is the NEXT timestep after the window
        target_volume = y_volume[i + window_size]
        target_congestion = y_congestion[i + window_size]
        
        X_windows.append(window)
        y_vol_windows.append(target_volume)
        y_cong_windows.append(target_congestion)
    
    X_windows = np.array(X_windows)  # (num_windows, 24, num_features)
    y_vol_windows = np.array(y_vol_windows)
    y_cong_windows = np.array(y_cong_windows)
    
    return X_windows, y_vol_windows, y_cong_windows

# Create windows for each split
X_train_windows, y_train_vol_windows, y_train_cong_windows = create_tcn_windows(
    X_train_scaled, y_train_volume, y_train_congestion
)

X_val_windows, y_val_vol_windows, y_val_cong_windows = create_tcn_windows(
    X_val_scaled, y_val_volume, y_val_congestion
)

X_test_windows, y_test_vol_windows, y_test_cong_windows = create_tcn_windows(
    X_test_scaled, y_test_volume, y_test_congestion
)

print(f"Training windows: {X_train_windows.shape}")
print(f"  - Shape: (num_windows={X_train_windows.shape[0]}, seq_len={X_train_windows.shape[1]}, features={X_train_windows.shape[2]})")
print(f"Validation windows: {X_val_windows.shape}")
print(f"Test windows: {X_test_windows.shape}")

# Check class distribution in labels
print(f"\nCongestion class distribution in training labels:")
unique, counts = np.unique(y_train_cong_windows, return_counts=True)
for label, count in zip(unique, counts):
    print(f"  Class {int(label)}: {count} samples ({100*count/len(y_train_cong_windows):.1f}%)")

Training windows: (636, 24, 22)
  - Shape: (num_windows=636, seq_len=24, features=22)
Validation windows: (139, 24, 22)
Test windows: (236, 24, 22)

Congestion class distribution in training labels:
  Class 0: 131 samples (20.6%)
  Class 1: 138 samples (21.7%)
  Class 2: 132 samples (20.8%)
  Class 3: 123 samples (19.3%)
  Class 4: 112 samples (17.6%)


In [10]:
# Check for NaN values in windows
print(f"X_train_windows has NaN: {np.isnan(X_train_windows).any()}")
print(f"X_val_windows has NaN: {np.isnan(X_val_windows).any()}")
print(f"X_test_windows has NaN: {np.isnan(X_test_windows).any()}")

nan_count_train = np.isnan(X_train_windows).sum()
print(f"\nNaN count in X_train_windows: {nan_count_train} out of {X_train_windows.size}")

if nan_count_train > 0:
    nan_window_indices = np.where(np.isnan(X_train_windows).any(axis=(1, 2)))[0]
    print(f"Number of windows with NaN: {len(nan_window_indices)}")
    print(f"First few NaN window indices: {nan_window_indices[:5]}")
else:
    print("No NaN values in training windows")

X_train_windows has NaN: False
X_val_windows has NaN: False
X_test_windows has NaN: False

NaN count in X_train_windows: 0 out of 335808
No NaN values in training windows


creating batch generator

In [11]:
class DataLoader:
    """Simple DataLoader implementation for batching"""
    def __init__(self, X, y_volume, y_congestion, batch_size=32, shuffle=False):
        self.X = X
        self.y_volume = y_volume
        self.y_congestion = y_congestion
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.n_samples = len(X)
        self.indices = np.arange(self.n_samples)
        
    def __iter__(self):
        """Iterate through batches"""
        if self.shuffle:
            np.random.shuffle(self.indices)
        
        for i in range(0, self.n_samples, self.batch_size):
            batch_indices = self.indices[i:i+self.batch_size]
            
            batch_X = self.X[batch_indices]
            batch_y_vol = self.y_volume[batch_indices]
            batch_y_cong = self.y_congestion[batch_indices]
            
            yield {
                'X': batch_X,
                'y_volume': batch_y_vol,
                'y_congestion': batch_y_cong,
                'indices': batch_indices
            }
    
    def __len__(self):
        """Number of batches"""
        return (self.n_samples + self.batch_size - 1) // self.batch_size

# Create dataloaders (NO shuffle for time series!)
batch_size = 32

train_loader = DataLoader(X_train_windows, y_train_vol_windows, y_train_cong_windows, 
                          batch_size=batch_size, shuffle=False)
val_loader = DataLoader(X_val_windows, y_val_vol_windows, y_val_cong_windows, 
                        batch_size=batch_size, shuffle=False)
test_loader = DataLoader(X_test_windows, y_test_vol_windows, y_test_cong_windows, 
                         batch_size=batch_size, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

# Verify a single batch
sample_batch = next(iter(train_loader))
print(f"\nSample batch shapes:")
print(f"  X: {sample_batch['X'].shape}")
print(f"  y_volume: {sample_batch['y_volume'].shape}")
print(f"  y_congestion: {sample_batch['y_congestion'].shape}")

# Save all data for next steps
data_package = {
    'X_train_windows': X_train_windows,
    'y_train_vol_windows': y_train_vol_windows,
    'y_train_cong_windows': y_train_cong_windows,
    'X_val_windows': X_val_windows,
    'y_val_vol_windows': y_val_vol_windows,
    'y_val_cong_windows': y_val_cong_windows,
    'X_test_windows': X_test_windows,
    'y_test_vol_windows': y_test_vol_windows,
    'y_test_cong_windows': y_test_cong_windows,
    'scaler_mean': scaler.mean.tolist(),
    'scaler_std': scaler.std.tolist(),
    'tcn_input_features': tcn_input_features,
    'num_features': X_train_windows.shape[2],
    'window_size': 24,
    'num_classes': len(np.unique(y_train_cong_windows))
}

# Save to JSON
with open('data_package.json', 'w') as f:
    json.dump({k: v for k, v in data_package.items() if not isinstance(v, np.ndarray)}, f)

# Save NumPy arrays
np.save('X_train_windows.npy', X_train_windows)
np.save('y_train_vol_windows.npy', y_train_vol_windows)
np.save('y_train_cong_windows.npy', y_train_cong_windows)
np.save('X_val_windows.npy', X_val_windows)
np.save('y_val_vol_windows.npy', y_val_vol_windows)
np.save('y_val_cong_windows.npy', y_val_cong_windows)
np.save('X_test_windows.npy', X_test_windows)
np.save('y_test_vol_windows.npy', y_test_vol_windows)
np.save('y_test_cong_windows.npy', y_test_cong_windows)


Train batches: 20
Val batches: 5
Test batches: 8

Sample batch shapes:
  X: (32, 24, 22)
  y_volume: (32,)
  y_congestion: (32,)


implementing TCN encoder architecture

In [12]:
class ResidualBlock(nn.Module):
    """
    PyTorch Residual block with causal dilated convolution.
    """
    def __init__(self, in_channels, out_channels, kernel_size=3, dilation=1, dropout=0.2):
        super().__init__()
        # Causal padding: pad on left only
        self.padding = (kernel_size - 1) * dilation
        
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size, 
                             padding=self.padding, dilation=dilation, padding_mode='zeros')
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        
        # Residual connection (1x1 conv if channels differ)
        if in_channels == out_channels:
            self.residual = nn.Identity()
        else:
            self.residual = nn.Conv1d(in_channels, out_channels, 1)
    
    def forward(self, x):
        """
        Args:
            x: (batch, seq_len, in_channels)
        Returns:
            output: (batch, seq_len, out_channels)
        """
        # Convert to (batch, in_channels, seq_len) for Conv1d
        x_t = x.transpose(1, 2)
        
        out = self.conv(x_t)
        # Remove right padding to maintain causal property
        out = out[:, :, :x_t.shape[-1]]
        
        out = self.relu(out)
        out = self.dropout(out)
        
        # Residual connection
        res = self.residual(x_t)
        out = res + out
        
        # Convert back to (batch, seq_len, out_channels)
        return out.transpose(1, 2)


class TCNEncoder(nn.Module):
    """
    PyTorch Temporal Convolutional Network encoder.
    
    Architecture:
    - 4 residual blocks with exponentially increasing dilation
    - Input: (batch, 24, num_features)
    - Output: (batch, 24, 64)
    """
    def __init__(self, input_dim=11, num_filters=64, kernel_size=3, 
                 dilations=[1, 2, 4, 8], dropout=0.2):
        super().__init__()
        
        self.input_dim = input_dim
        self.num_filters = num_filters
        
        # Build sequence of residual blocks
        blocks = []
        for i, dilation in enumerate(dilations):
            in_ch = input_dim if i == 0 else num_filters
            out_ch = num_filters
            
            blocks.append(ResidualBlock(
                in_channels=in_ch,
                out_channels=out_ch,
                kernel_size=kernel_size,
                dilation=dilation,
                dropout=dropout
            ))
        
        self.blocks = nn.Sequential(*blocks)
        
        print(f"TCN Encoder (PyTorch):")
        print(f"  Input dimension: {input_dim}")
        print(f"  4 Residual blocks (dilations: {dilations})")
        print(f"  Output: (batch, 24, {num_filters})")
    
    def forward(self, x):
        """
        Args:
            x: (batch, 24, num_features)
        Returns:
            output: (batch, 24, 64)
        """
        return self.blocks(x)


print("TCN Encoder created with:")
print(f"  Input dimension: {data_package['num_features']}")
print(f"  4 dilated blocks: [1, 2, 4, 8]")
print(f"  Output filters: 64")
print(f"  Window size: 24 hours")

# Instantiate encoder
encoder = TCNEncoder(
    input_dim=data_package['num_features'],
    num_filters=64,
    kernel_size=3,
    dilations=[1, 2, 4, 8],
    dropout=0.2
)


TCN Encoder created with:
  Input dimension: 22
  4 dilated blocks: [1, 2, 4, 8]
  Output filters: 64
  Window size: 24 hours
TCN Encoder (PyTorch):
  Input dimension: 22
  4 Residual blocks (dilations: [1, 2, 4, 8])
  Output: (batch, 24, 64)


implementing masked auto encoder loss

In [13]:
class DecoderHead(nn.Module):
    """PyTorch decoder to reconstruct original features from encoded output"""
    
    def __init__(self, encoded_dim=64, output_dim=11):
        super().__init__()
        self.linear = nn.Linear(encoded_dim, output_dim)
    
    def forward(self, x_encoded):
        """
        Decode encoded features.
        
        Args:
            x_encoded: (batch, seq_len, encoded_dim)
        
        Returns:
            reconstructed: (batch, seq_len, output_dim)
        """
        # Linear transformation: (batch, seq_len, encoded_dim) -> (batch, seq_len, output_dim)
        return self.linear(x_encoded)


def create_mask(X, mask_probability=0.2):
    """
    Create random mask for 20% of sequence positions.
    
    Args:
        X: (batch, seq_len, features)
        mask_probability: probability of masking (default 0.2)
    
    Returns:
        mask: (batch, seq_len) boolean array
    """
    batch_size, seq_len = X.shape[0], X.shape[1]
    mask = np.random.rand(batch_size, seq_len) < mask_probability
    return mask


def apply_mask(X, mask, mask_value=0.0):
    """
    Apply mask to input by replacing masked positions with mask_value.
    
    Args:
        X: (batch, seq_len, features)
        mask: (batch, seq_len) boolean
        mask_value: value to replace masked positions
    
    Returns:
        X_masked: (batch, seq_len, features)
    """
    X_masked = X.copy()
    for b in range(X.shape[0]):
        for t in range(X.shape[1]):
            if mask[b, t]:
                X_masked[b, t, :] = mask_value
    return X_masked


def mse_loss(y_pred, y_true, mask=None):
    """
    Compute MSE loss.
    If mask is provided, compute loss only on masked positions.
    
    Args:
        y_pred: predicted values (batch, seq_len, features)
        y_true: true values (batch, seq_len, features)
        mask: (batch, seq_len) boolean mask (True = compute loss)
    
    Returns:
        loss: scalar mean squared error
    """
    if mask is None:
        # Compute loss on all positions
        return np.mean((y_pred - y_true) ** 2)
    else:
        # Compute loss only on masked positions
        diff = (y_pred - y_true) ** 2
        masked_diff = np.zeros_like(diff)
        
        for b in range(diff.shape[0]):
            for t in range(diff.shape[1]):
                if mask[b, t]:
                    masked_diff[b, t, :] = diff[b, t, :]
        
        return np.mean(masked_diff)


class Phase1Model(nn.Module):
    """
    Phase 1 model: TCN encoder + reconstruction head (PyTorch).
    """
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
    
    def forward(self, x):
        """Forward pass: x -> encoder -> decoder -> reconstructed"""
        encoded = self.encoder(x)  # (batch, 24, 64)
        reconstructed = self.decoder(encoded)  # (batch, 24, features)
        return reconstructed, encoded


# Move encoder to PyTorch and initialize model
encoder_pt = TCNEncoder(
    input_dim=data_package['num_features'],
    num_filters=64,
    kernel_size=3,
    dilations=[1, 2, 4, 8],
    dropout=0.2
)

decoder_pt = DecoderHead(encoded_dim=64, output_dim=data_package['num_features'])
phase1_model = Phase1Model(encoder_pt, decoder_pt)

# Move to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
phase1_model = phase1_model.to(device)

print(f"\nModel device: {device}")
print(f"Total parameters: {sum(p.numel() for p in phase1_model.parameters()):,}")

TCN Encoder (PyTorch):
  Input dimension: 22
  4 Residual blocks (dilations: [1, 2, 4, 8])
  Output: (batch, 24, 64)

Model device: cpu
Total parameters: 44,246


trianing for phase one

In [14]:
def train_phase1(model, train_loader, val_loader, num_epochs=100, learning_rate=1e-3, 
                patience=10, device='cpu', verbose=True):
    """
    Train Phase 1 with PyTorch: Self-supervised reconstruction.
    
    Args:
        model: Phase1Model instance
        train_loader: training data loader
        val_loader: validation data loader
        num_epochs: maximum epochs
        learning_rate: learning rate
        patience: early stopping patience
        device: torch device
        verbose: print progress
    
    Returns:
        train_losses: list of training losses per epoch
        val_losses: list of validation losses per epoch
    """
    
    train_losses = []
    val_losses = []
    best_val_loss = float('inf')
    epochs_no_improve = 0
    
    # Optimizer and loss
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.MSELoss()
    
    if verbose:
        print("\n" + "="*80)
        print("PHASE 1: Self-Supervised Representation Learning (PyTorch)")
        print("="*80)
        print(f"Optimizer: Adam")
        print(f"Learning rate: {learning_rate}")
        print(f"Mask probability: 0.2 (mask 20% of sequence)")
        print(f"Early stopping patience: {patience}")
        print(f"Device: {device}")
        print("="*80 + "\n")
    
    for epoch in range(num_epochs):
        # ============= TRAINING =============
        model.train()
        train_loss = 0.0
        num_batches = 0
        
        for batch in train_loader:
            X_batch = batch['X']  # (batch, 24, features)
            
            # Create mask and apply
            mask = create_mask(X_batch, mask_probability=0.2)
            X_masked = apply_mask(X_batch, mask, mask_value=0.0)
            
            # Convert to tensors
            X_tensor = torch.tensor(X_masked, dtype=torch.float32).to(device)
            X_true = torch.tensor(X_batch, dtype=torch.float32).to(device)
            mask_tensor = torch.tensor(mask, dtype=torch.bool).to(device)
            
            # Forward pass
            reconstructed, encoded = model(X_tensor)
            
            # Loss: MSE on masked positions only
            # Expand mask to match feature dimension
            mask_expanded = mask_tensor.unsqueeze(-1).expand_as(reconstructed).float()
            
            # Compute squared difference and apply mask
            squared_diff = (reconstructed - X_true) ** 2
            masked_loss = (squared_diff * mask_expanded).sum() / (mask_expanded.sum() + 1e-8)
            loss = masked_loss
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            train_loss += loss.item()
            num_batches += 1
        
        train_loss /= num_batches
        train_losses.append(train_loss)
        
        # ============= VALIDATION =============
        model.eval()
        val_loss = 0.0
        num_batches = 0
        
        with torch.no_grad():
            for batch in val_loader:
                X_batch = batch['X']
                
                mask = create_mask(X_batch, mask_probability=0.2)
                X_masked = apply_mask(X_batch, mask, mask_value=0.0)
                
                X_tensor = torch.tensor(X_masked, dtype=torch.float32).to(device)
                X_true = torch.tensor(X_batch, dtype=torch.float32).to(device)
                mask_tensor = torch.tensor(mask, dtype=torch.bool).to(device)
                
                reconstructed, encoded = model(X_tensor)
                
                mask_expanded = mask_tensor.unsqueeze(-1).expand_as(reconstructed).float()
                
                # Compute squared difference and apply mask
                squared_diff = (reconstructed - X_true) ** 2
                masked_loss = (squared_diff * mask_expanded).sum() / (mask_expanded.sum() + 1e-8)
                loss = masked_loss
                
                val_loss += loss.item()
                num_batches += 1
        
        val_loss /= num_batches
        val_losses.append(val_loss)
        
        # ============= LOGGING =============
        if verbose and (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.6f} | Val Loss: {val_loss:.6f}")
        
        # ============= EARLY STOPPING =============
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            
            # Save checkpoint
            checkpoint = {
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'train_loss': train_loss,
                'val_loss': val_loss
            }
            
            torch.save(checkpoint, 'phase1_checkpoint.pt')
            
            if verbose and (epoch + 1) % 10 == 0:
                print(f"  ✓ Best val loss! Saved checkpoint")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                if verbose:
                    print(f"\nEarly stopping at epoch {epoch+1}")
                    print(f"Best val loss: {best_val_loss:.6f}")
                break
    
    if verbose:
        print("\n" + "="*80)
        print(f"Phase 1 Training Complete")
        print(f"Best validation loss: {best_val_loss:.6f}")
        print("="*80 + "\n")
    
    return train_losses, val_losses


# Create data loaders
train_loader = DataLoader(X_train_windows, y_train_vol_windows, y_train_cong_windows, 
                          batch_size=32, shuffle=False)
val_loader = DataLoader(X_val_windows, y_val_vol_windows, y_val_cong_windows, 
                        batch_size=32, shuffle=False)

# Train Phase 1 with PyTorch
train_losses_phase1, val_losses_phase1 = train_phase1(
    model=phase1_model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=100,
    learning_rate=1e-3,
    patience=10,
    device=device,
    verbose=True
)


PHASE 1: Self-Supervised Representation Learning (PyTorch)
Optimizer: Adam
Learning rate: 0.001
Mask probability: 0.2 (mask 20% of sequence)
Early stopping patience: 10
Device: cpu

Epoch 10/100 | Train Loss: 0.704684 | Val Loss: 0.710116
  ✓ Best val loss! Saved checkpoint
Epoch 20/100 | Train Loss: 0.627202 | Val Loss: 0.802460

Early stopping at epoch 27
Best val loss: 0.684178

Phase 1 Training Complete
Best validation loss: 0.684178



In [15]:
# Debug: Check data shapes and initial loss
batch = next(iter(train_loader))
X_batch = batch['X']
print(f"X_batch shape: {X_batch.shape}, dtype: {X_batch.dtype}")
print(f"X_batch min/max: {X_batch.min():.4f} / {X_batch.max():.4f}")
print(f"X_batch has NaN: {np.isnan(X_batch).any()}")

mask = create_mask(X_batch, mask_probability=0.2)
X_masked = apply_mask(X_batch, mask, mask_value=0.0)
print(f"\nX_masked min/max: {X_masked.min():.4f} / {X_masked.max():.4f}")
print(f"X_masked has NaN: {np.isnan(X_masked).any()}")

X_tensor = torch.tensor(X_masked, dtype=torch.float32).to(device)
X_true = torch.tensor(X_batch, dtype=torch.float32).to(device)
print(f"\nX_tensor: shape={X_tensor.shape}, dtype={X_tensor.dtype}")
print(f"X_true: shape={X_true.shape}, dtype={X_true.dtype}")

reconstructed, encoded = phase1_model(X_tensor)
print(f"\nReconstructed: shape={reconstructed.shape}, dtype={reconstructed.dtype}")
print(f"Reconstructed has NaN: {torch.isnan(reconstructed).any()}")
print(f"Reconstructed min/max: {reconstructed.min():.4f} / {reconstructed.max():.4f}")

mask_tensor = torch.tensor(mask, dtype=torch.bool).to(device)
mask_expanded = mask_tensor.unsqueeze(-1).expand_as(reconstructed).float()
squared_diff = (reconstructed - X_true) ** 2
print(f"\nSquared diff: shape={squared_diff.shape}, has NaN={torch.isnan(squared_diff).any()}")
print(f"Squared diff min/max: {squared_diff.min():.6f} / {squared_diff.max():.6f}")

masked_loss = (squared_diff * mask_expanded).sum() / (mask_expanded.sum() + 1e-8)
print(f"Masked loss: {masked_loss.item():.6f}")
print(f"Loss is NaN: {torch.isnan(masked_loss)}")

X_batch shape: (32, 24, 22), dtype: float64
X_batch min/max: -2.4554 / 25.6710
X_batch has NaN: False

X_masked min/max: -2.4554 / 25.6710
X_masked has NaN: False

X_tensor: shape=torch.Size([32, 24, 22]), dtype=torch.float32
X_true: shape=torch.Size([32, 24, 22]), dtype=torch.float32

Reconstructed: shape=torch.Size([32, 24, 22]), dtype=torch.float32
Reconstructed has NaN: False
Reconstructed min/max: -4.0774 / 15.0291

Squared diff: shape=torch.Size([32, 24, 22]), has NaN=False
Squared diff min/max: 0.000000 / 717.647034
Masked loss: 0.616547
Loss is NaN: False


Loading best phase 1 checkpoint and freezing encoder

In [16]:
# Load Phase 1 checkpoint
checkpoint = torch.load('phase1_checkpoint.pt', map_location=device)

print(f"Loaded Phase 1 checkpoint (epoch {checkpoint['epoch']+1})")
print(f"  Train loss: {checkpoint['train_loss']:.6f}")
print(f"  Val loss: {checkpoint['val_loss']:.6f}")

# Restore model weights from checkpoint
phase1_model.load_state_dict(checkpoint['model_state_dict'])
phase1_model.to(device)
phase1_model.eval()

# Freeze encoder parameters (for transfer learning)
for param in phase1_model.encoder.parameters():
    param.requires_grad = False

print("\nEncoder frozen for Phase 2 transfer learning:")
for name, param in phase1_model.encoder.named_parameters():
    print(f"  {name}: requires_grad = {param.requires_grad}")

Loaded Phase 1 checkpoint (epoch 17)
  Train loss: 0.652261
  Val loss: 0.684178

Encoder frozen for Phase 2 transfer learning:
  blocks.0.conv.weight: requires_grad = False
  blocks.0.conv.bias: requires_grad = False
  blocks.0.residual.weight: requires_grad = False
  blocks.0.residual.bias: requires_grad = False
  blocks.1.conv.weight: requires_grad = False
  blocks.1.conv.bias: requires_grad = False
  blocks.2.conv.weight: requires_grad = False
  blocks.2.conv.bias: requires_grad = False
  blocks.3.conv.weight: requires_grad = False
  blocks.3.conv.bias: requires_grad = False


saving encoder for phase two

In [17]:
# Extract encoder as feature extractor
encoder_for_phase2 = phase1_model.encoder  # Already frozen

# Create Phase 2 task heads
class RegressionHead(nn.Module):
    """Regression head for vehicle count prediction"""
    def __init__(self, encoded_dim=64, output_dim=1):
        super().__init__()
        self.fc1 = nn.Linear(encoded_dim, 32)
        self.fc2 = nn.Linear(32, output_dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
    
    def forward(self, x_encoded):
        """
        Args:
            x_encoded: (batch, 24, 64) encoder output
        Returns:
            vol_pred: (batch, 1) vehicle count prediction
        """
        # Global average pooling over sequence
        x_pooled = x_encoded.mean(dim=1)  # (batch, 64)
        
        out = self.fc1(x_pooled)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)  # (batch, 1)
        return out


class ClassificationHead(nn.Module):
    """Classification head for congestion level prediction"""
    def __init__(self, encoded_dim=64, num_classes=5):
        super().__init__()
        self.fc1 = nn.Linear(encoded_dim, 32)
        self.fc2 = nn.Linear(32, num_classes)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
    
    def forward(self, x_encoded):
        """
        Args:
            x_encoded: (batch, 24, 64) encoder output
        Returns:
            cong_logits: (batch, 5) congestion level logits
        """
        # Global average pooling over sequence
        x_pooled = x_encoded.mean(dim=1)  # (batch, 64)
        
        out = self.fc1(x_pooled)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)  # (batch, 5)
        return out


# Instantiate phase 2 heads
regression_head = RegressionHead(encoded_dim=64, output_dim=1)
classification_head = ClassificationHead(encoded_dim=64, num_classes=5)

# Move to device
regression_head = regression_head.to(device)
classification_head = classification_head.to(device)

print("Phase 2 task heads created:")
print(f"  Regression head (vehicle count): {sum(p.numel() for p in regression_head.parameters())} parameters")
print(f"  Classification head (congestion): {sum(p.numel() for p in classification_head.parameters())} parameters")
print(f"\nTotal Phase 2 trainable parameters: {sum(p.numel() for p in list(regression_head.parameters()) + list(classification_head.parameters()))}")
print(f"Frozen encoder parameters: {sum(p.numel() for p in encoder_for_phase2.parameters())}")

Phase 2 task heads created:
  Regression head (vehicle count): 2113 parameters
  Classification head (congestion): 2245 parameters

Total Phase 2 trainable parameters: 4358
Frozen encoder parameters: 42816


Phase 2 - Fine-tuning downstream task heads

setting up Phase 2 training

In [18]:
# Data is already loaded in memory from Phase 1
# Create new DataLoaders for Phase 2 training (using same data as Phase 1)
phase2_train_loader = DataLoader(X_train_windows, y_train_vol_windows, y_train_cong_windows, 
                                  batch_size=32, shuffle=False)
phase2_val_loader = DataLoader(X_val_windows, y_val_vol_windows, y_val_cong_windows, 
                                batch_size=32, shuffle=False)
phase2_test_loader = DataLoader(X_test_windows, y_test_vol_windows, y_test_cong_windows, 
                                 batch_size=32, shuffle=False)

print("Phase 2 DataLoaders created:")
print(f"  Train batches: {len(phase2_train_loader)}")
print(f"  Val batches: {len(phase2_val_loader)}")
print(f"  Test batches: {len(phase2_test_loader)}")
print(f"\nFrozen encoder from Phase 1: {sum(p.numel() for p in encoder_for_phase2.parameters()):,} parameters")
print(f"Regression head: {sum(p.numel() for p in regression_head.parameters()):,} parameters")
print(f"Classification head: {sum(p.numel() for p in classification_head.parameters()):,} parameters")

Phase 2 DataLoaders created:
  Train batches: 20
  Val batches: 5
  Test batches: 8

Frozen encoder from Phase 1: 42,816 parameters
Regression head: 2,113 parameters
Classification head: 2,245 parameters


implementing projection head

In [19]:
# Phase 2 heads are already created in previous cells
# Just verify they are properly initialized

print("Phase 2 Model Architecture Summary:")
print("\n" + "="*80)
print("FROZEN ENCODER (from Phase 1)")
print("="*80)
print(f"Total parameters: {sum(p.numel() for p in encoder_for_phase2.parameters()):,}")
print(f"Trainable: False (all requires_grad=False)")

print("\n" + "="*80)
print("REGRESSION HEAD (vehicle count)")
print("="*80)
print(f"Input: (batch, 24, 64) from encoder")
print(f"Architecture:")
for name, module in regression_head.named_modules():
    if isinstance(module, nn.Linear):
        print(f"  {name}: in={module.in_features}, out={module.out_features}")
    elif isinstance(module, nn.ReLU):
        print(f"  {name}: ReLU()")
    elif isinstance(module, nn.Dropout):
        print(f"  {name}: Dropout(p={module.p})")
print(f"Output: (batch, 1) vehicle count prediction")
print(f"Total parameters: {sum(p.numel() for p in regression_head.parameters()):,}")
print(f"Trainable: True")

print("\n" + "="*80)
print("CLASSIFICATION HEAD (congestion level)")
print("="*80)
print(f"Input: (batch, 24, 64) from encoder")
print(f"Architecture:")
for name, module in classification_head.named_modules():
    if isinstance(module, nn.Linear):
        print(f"  {name}: in={module.in_features}, out={module.out_features}")
    elif isinstance(module, nn.ReLU):
        print(f"  {name}: ReLU()")
    elif isinstance(module, nn.Dropout):
        print(f"  {name}: Dropout(p={module.p})")
print(f"Output: (batch, 5) congestion logits (5 classes)")
print(f"Total parameters: {sum(p.numel() for p in classification_head.parameters()):,}")
print(f"Trainable: True")

print("\n" + "="*80)
print("PHASE 2 TRAINING CONFIGURATION")
print("="*80)
print(f"Encoder: FROZEN (requires_grad=False)")
print(f"Task heads: TRAINABLE (requires_grad=True)")
print(f"Training strategy: Transfer learning with frozen feature extractor")
print("="*80 + "\n")

Phase 2 Model Architecture Summary:

FROZEN ENCODER (from Phase 1)
Total parameters: 42,816
Trainable: False (all requires_grad=False)

REGRESSION HEAD (vehicle count)
Input: (batch, 24, 64) from encoder
Architecture:
  fc1: in=64, out=32
  fc2: in=32, out=1
  relu: ReLU()
  dropout: Dropout(p=0.3)
Output: (batch, 1) vehicle count prediction
Total parameters: 2,113
Trainable: True

CLASSIFICATION HEAD (congestion level)
Input: (batch, 24, 64) from encoder
Architecture:
  fc1: in=64, out=32
  fc2: in=32, out=5
  relu: ReLU()
  dropout: Dropout(p=0.3)
Output: (batch, 5) congestion logits (5 classes)
Total parameters: 2,245
Trainable: True

PHASE 2 TRAINING CONFIGURATION
Encoder: FROZEN (requires_grad=False)
Task heads: TRAINABLE (requires_grad=True)
Training strategy: Transfer learning with frozen feature extractor



implementing multi task losss function

In [20]:
# Phase 2 Loss Functions (PyTorch)

class Phase2Loss(nn.Module):
    """
    Multi-task loss for Phase 2:
    - Regression: MAE for vehicle count (more robust to outliers than MSE)
    - Classification: CrossEntropyLoss for congestion level
    
    Combined: α·MAE(regression) + β·CrossEntropy(classification)
    """
    
    def __init__(self, alpha=0.7, beta=0.3):
        super().__init__()
        self.alpha = alpha  # Regression weight
        self.beta = beta    # Classification weight
        
        self.mae_loss = nn.L1Loss()
        self.ce_loss = nn.CrossEntropyLoss()
    
    def forward(self, vol_pred, vol_true, cong_logits, cong_true):
        """
        Args:
            vol_pred: (batch, 1) predicted vehicle count
            vol_true: (batch,) or (batch, 1) true vehicle count
            cong_logits: (batch, 5) congestion logits
            cong_true: (batch,) congestion class indices (0-4)
        
        Returns:
            loss: scalar total loss
            loss_dict: dict with individual loss components
        """
        
        # Ensure correct shapes
        if vol_true.dim() > 1:
            vol_true = vol_true.squeeze()
        if vol_pred.dim() > 1:
            vol_pred = vol_pred.squeeze()
        
        # Regression loss
        loss_regression = self.mae_loss(vol_pred, vol_true.float())
        
        # Classification loss
        loss_classification = self.ce_loss(cong_logits, cong_true.long())
        
        # Combined loss
        loss_total = self.alpha * loss_regression + self.beta * loss_classification
        
        return loss_total, {
            'total': loss_total.item(),
            'regression_mae': loss_regression.item(),
            'classification_ce': loss_classification.item()
        }


# Instantiate loss function
phase2_loss = Phase2Loss(alpha=0.7, beta=0.3)

print("Phase 2 Loss Function:")
print("="*80)
print("Multi-task Loss = 0.7×MAE(vehicle_count) + 0.3×CrossEntropy(congestion)")
print("="*80)
print(f"\nRegression Loss: MAE (more robust to outliers)")
print(f"  Input: (batch,) predicted vs (batch,) true vehicle counts")
print(f"  Weight (α): 0.7")

print(f"\nClassification Loss: CrossEntropyLoss (with softmax)")
print(f"  Input: (batch, 5) logits vs (batch,) class indices")
print(f"  Weight (β): 0.3")
print(f"\nOptimization: Only regression_head and classification_head parameters\n")

Phase 2 Loss Function:
Multi-task Loss = 0.7×MAE(vehicle_count) + 0.3×CrossEntropy(congestion)

Regression Loss: MAE (more robust to outliers)
  Input: (batch,) predicted vs (batch,) true vehicle counts
  Weight (α): 0.7

Classification Loss: CrossEntropyLoss (with softmax)
  Input: (batch, 5) logits vs (batch,) class indices
  Weight (β): 0.3

Optimization: Only regression_head and classification_head parameters



training loop fpr phase two

In [21]:
def train_phase2(encoder, regression_head, classification_head, loss_fn, 
                train_loader, val_loader, num_epochs=50, learning_rate=1e-4, 
                patience=10, device='cpu', verbose=True):
    """
    Train Phase 2: Fine-tune task heads with frozen encoder.
    
    Args:
        encoder: Frozen feature extractor from Phase 1
        regression_head: RegressionHead for vehicle count
        classification_head: ClassificationHead for congestion
        loss_fn: Phase2Loss instance
        train_loader: Training DataLoader
        val_loader: Validation DataLoader
        num_epochs: Maximum epochs
        learning_rate: Learning rate for head parameters only
        patience: Early stopping patience
        device: torch device
        verbose: Print progress
    
    Returns:
        train_history: dict with loss lists
        val_history: dict with loss lists
    """
    
    # Encoder is frozen - only optimize head parameters
    params_to_optimize = (
        list(regression_head.parameters()) + 
        list(classification_head.parameters())
    )
    
    optimizer = optim.Adam(params_to_optimize, lr=learning_rate)
    
    train_history = {
        'total': [],
        'regression_mae': [],
        'classification_ce': []
    }
    val_history = {
        'total': [],
        'regression_mae': [],
        'classification_ce': []
    }
    
    best_val_loss = float('inf')
    epochs_no_improve = 0
    
    if verbose:
        print("\n" + "="*80)
        print("PHASE 2: Fine-tuning Downstream Task Heads")
        print("="*80)
        print(f"Frozen encoder: TCNEncoder (53,876 parameters)")
        print(f"Trainable regression head: {sum(p.numel() for p in regression_head.parameters()):,} parameters")
        print(f"Trainable classification head: {sum(p.numel() for p in classification_head.parameters()):,} parameters")
        print(f"Optimizer: Adam")
        print(f"Learning rate: {learning_rate}")
        print(f"Loss: 0.7×MAE(vehicle_count) + 0.3×CrossEntropy(congestion)")
        print(f"Early stopping patience: {patience}")
        print(f"Device: {device}")
        print("="*80 + "\n")
    
    for epoch in range(num_epochs):
        # ============= TRAINING =============
        encoder.eval()  # Frozen encoder stays in eval mode
        regression_head.train()
        classification_head.train()
        
        train_total = 0.0
        train_reg = 0.0
        train_clf = 0.0
        num_batches = 0
        
        for batch in train_loader:
            X_batch = batch['X']
            y_vol_batch = batch['y_volume']
            y_cong_batch = batch['y_congestion']
            
            # Convert to tensors
            X_tensor = torch.tensor(X_batch, dtype=torch.float32).to(device)
            y_vol_tensor = torch.tensor(y_vol_batch, dtype=torch.float32).to(device)
            y_cong_tensor = torch.tensor(y_cong_batch, dtype=torch.long).to(device)
            
            # Forward pass
            with torch.no_grad():
                encoded = encoder(X_tensor)  # (batch, 24, 64)
            
            vol_pred = regression_head(encoded)  # (batch, 1)
            cong_logits = classification_head(encoded)  # (batch, 5)
            
            # Loss computation
            loss_total, loss_dict = loss_fn(vol_pred, y_vol_tensor, 
                                            cong_logits, y_cong_tensor)
            
            # Backward pass
            optimizer.zero_grad()
            loss_total.backward()
            torch.nn.utils.clip_grad_norm_(params_to_optimize, max_norm=1.0)
            optimizer.step()
            
            train_total += loss_dict['total']
            train_reg += loss_dict['regression_mae']
            train_clf += loss_dict['classification_ce']
            num_batches += 1
        
        train_total /= num_batches
        train_reg /= num_batches
        train_clf /= num_batches
        
        train_history['total'].append(train_total)
        train_history['regression_mae'].append(train_reg)
        train_history['classification_ce'].append(train_clf)
        
        # ============= VALIDATION =============
        encoder.eval()
        regression_head.eval()
        classification_head.eval()
        
        val_total = 0.0
        val_reg = 0.0
        val_clf = 0.0
        num_batches = 0
        
        with torch.no_grad():
            for batch in val_loader:
                X_batch = batch['X']
                y_vol_batch = batch['y_volume']
                y_cong_batch = batch['y_congestion']
                
                X_tensor = torch.tensor(X_batch, dtype=torch.float32).to(device)
                y_vol_tensor = torch.tensor(y_vol_batch, dtype=torch.float32).to(device)
                y_cong_tensor = torch.tensor(y_cong_batch, dtype=torch.long).to(device)
                
                encoded = encoder(X_tensor)
                vol_pred = regression_head(encoded)
                cong_logits = classification_head(encoded)
                
                loss_total, loss_dict = loss_fn(vol_pred, y_vol_tensor, 
                                                cong_logits, y_cong_tensor)
                
                val_total += loss_dict['total']
                val_reg += loss_dict['regression_mae']
                val_clf += loss_dict['classification_ce']
                num_batches += 1
        
        val_total /= num_batches
        val_reg /= num_batches
        val_clf /= num_batches
        
        val_history['total'].append(val_total)
        val_history['regression_mae'].append(val_reg)
        val_history['classification_ce'].append(val_clf)
        
        # ============= LOGGING =============
        if verbose and (epoch + 1) % 10 == 0:
            print(f"Epoch {epoch+1}/{num_epochs}")
            print(f"  Train | Total: {train_total:.6f} | MAE: {train_reg:.6f} | CE: {train_clf:.6f}")
            print(f"  Val   | Total: {val_total:.6f} | MAE: {val_reg:.6f} | CE: {val_clf:.6f}")
        
        # ============= EARLY STOPPING =============
        if val_total < best_val_loss:
            best_val_loss = val_total
            epochs_no_improve = 0
            
            # Save checkpoint
            checkpoint = {
                'epoch': epoch,
                'regression_head_state': regression_head.state_dict(),
                'classification_head_state': classification_head.state_dict(),
                'optimizer_state': optimizer.state_dict(),
                'train_loss': train_total,
                'val_loss': val_total
            }
            
            torch.save(checkpoint, 'phase2_checkpoint.pt')
            
            if verbose and (epoch + 1) % 10 == 0:
                print(f"  ✓ Best val loss! Saved checkpoint")
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                if verbose:
                    print(f"\nEarly stopping at epoch {epoch+1}")
                    print(f"Best val loss: {best_val_loss:.6f}")
                break
    
    if verbose:
        print("\n" + "="*80)
        print(f"Phase 2 Training Complete")
        print(f"Best validation loss: {best_val_loss:.6f}")
        print("="*80 + "\n")
    
    return train_history, val_history


# Train Phase 2
print("Starting Phase 2 training...")
train_history_phase2, val_history_phase2 = train_phase2(
    encoder=encoder_for_phase2,
    regression_head=regression_head,
    classification_head=classification_head,
    loss_fn=phase2_loss,
    train_loader=phase2_train_loader,
    val_loader=phase2_val_loader,
    num_epochs=50,
    learning_rate=1e-4,
    patience=10,
    device=device,
    verbose=True
)

Starting Phase 2 training...

PHASE 2: Fine-tuning Downstream Task Heads
Frozen encoder: TCNEncoder (53,876 parameters)
Trainable regression head: 2,113 parameters
Trainable classification head: 2,245 parameters
Optimizer: Adam
Learning rate: 0.0001
Loss: 0.7×MAE(vehicle_count) + 0.3×CrossEntropy(congestion)
Early stopping patience: 10
Device: cpu

Epoch 10/50
  Train | Total: 13.678636 | MAE: 18.845923 | CE: 1.621632
  Val   | Total: 13.962313 | MAE: 19.253925 | CE: 1.615220
  ✓ Best val loss! Saved checkpoint
Epoch 20/50
  Train | Total: 12.529353 | MAE: 17.212460 | CE: 1.602105
  Val   | Total: 12.740912 | MAE: 17.510746 | CE: 1.611301
  ✓ Best val loss! Saved checkpoint
Epoch 30/50
  Train | Total: 11.024007 | MAE: 15.060841 | CE: 1.604728
  Val   | Total: 11.274100 | MAE: 15.416842 | CE: 1.607705
  ✓ Best val loss! Saved checkpoint
Epoch 40/50
  Train | Total: 9.371269 | MAE: 12.697689 | CE: 1.609622
  Val   | Total: 9.563597 | MAE: 12.974446 | CE: 1.604951
  ✓ Best val loss! Save

load best phase two checkpoint

In [22]:
# Load best Phase 2 checkpoint
checkpoint = torch.load('phase2_checkpoint.pt', map_location=device)

print(f"Loaded Phase 2 checkpoint (epoch {checkpoint['epoch']+1})")
print(f"  Train loss: {checkpoint['train_loss']:.6f}")
print(f"  Val loss: {checkpoint['val_loss']:.6f}")

# Restore best model weights
regression_head.load_state_dict(checkpoint['regression_head_state'])
classification_head.load_state_dict(checkpoint['classification_head_state'])

# Move to device and set to eval mode
regression_head = regression_head.to(device)
classification_head = classification_head.to(device)
regression_head.eval()
classification_head.eval()

print("\nModel restored to eval mode for testing")


Loaded Phase 2 checkpoint (epoch 50)
  Train loss: 7.521742
  Val loss: 7.626037

Model restored to eval mode for testing


evaluating  full TCN on test set

In [23]:
def evaluate_phase2(encoder, regression_head, classification_head, test_loader, 
                    device='cpu', verbose=True):
    """
    Evaluate Phase 2 model on test set.
    
    Metrics:
    - Regression: MAE, RMSE on vehicle count
    - Classification: Accuracy, Macro-F1 on congestion level
    """
    
    encoder.eval()
    regression_head.eval()
    classification_head.eval()
    
    volume_preds = []
    volume_trues = []
    congestion_preds = []
    congestion_trues = []
    
    with torch.no_grad():
        for batch in test_loader:
            X_batch = batch['X']
            y_vol_batch = batch['y_volume']
            y_cong_batch = batch['y_congestion']
            
            X_tensor = torch.tensor(X_batch, dtype=torch.float32).to(device)
            
            # Forward pass
            encoded = encoder(X_tensor)  # (batch, 24, 64)
            vol_pred = regression_head(encoded)  # (batch, 1)
            cong_logits = classification_head(encoded)  # (batch, 5)
            
            # Collect predictions
            volume_preds.append(vol_pred.squeeze().cpu().numpy())
            volume_trues.append(y_vol_batch)
            
            congestion_preds.append(cong_logits.argmax(dim=1).cpu().numpy())
            congestion_trues.append(y_cong_batch)
    
    # Concatenate all predictions
    volume_preds = np.concatenate(volume_preds)
    volume_trues = np.concatenate(volume_trues)
    congestion_preds = np.concatenate(congestion_preds)
    congestion_trues = np.concatenate(congestion_trues)
    
    # Regression metrics
    mae = np.mean(np.abs(volume_preds - volume_trues))
    rmse = np.sqrt(np.mean((volume_preds - volume_trues) ** 2))
    
    # Classification metrics
    accuracy = np.mean(congestion_preds == congestion_trues)
    
    # Macro-F1 score
    from sklearn.metrics import f1_score as sk_f1_score
    macro_f1 = sk_f1_score(congestion_trues, congestion_preds, average='macro', zero_division=0)
    
    metrics = {
        'mae': mae,
        'rmse': rmse,
        'accuracy': accuracy,
        'macro_f1': macro_f1,
        'volume_preds': volume_preds,
        'volume_trues': volume_trues,
        'congestion_preds': congestion_preds,
        'congestion_trues': congestion_trues
    }
    
    if verbose:
        print("\n" + "="*80)
        print("TEST SET EVALUATION - PHASE 2 MODEL")
        print("="*80)
        print(f"\nRegression (Vehicle Count):")
        print(f"  MAE:  {mae:.4f}")
        print(f"  RMSE: {rmse:.4f}")
        print(f"\nClassification (Congestion Level 0-4):")
        print(f"  Accuracy: {accuracy:.4f}")
        print(f"  Macro-F1: {macro_f1:.4f}")
        print("="*80 + "\n")
    
    return metrics


# Evaluate on test set
print("Evaluating Phase 2 model on test set...")
phase2_test_metrics = evaluate_phase2(
    encoder=encoder_for_phase2,
    regression_head=regression_head,
    classification_head=classification_head,
    test_loader=phase2_test_loader,
    device=device,
    verbose=True
)

Evaluating Phase 2 model on test set...

TEST SET EVALUATION - PHASE 2 MODEL

Regression (Vehicle Count):
  MAE:  9.5067
  RMSE: 10.5221

Classification (Congestion Level 0-4):
  Accuracy: 0.1822
  Macro-F1: 0.1549



abilation studies and error analysis

In [24]:
# Ablation Study: Compare Phase 2 Fine-tuned Model vs Phase 1 Encoder Only


print("\nModel A: Phase 1 Encoder Only (No fine-tuning)")
print("  - Uses frozen encoder + random task heads")
print("  - Represents baseline pre-trained features")

# Create fresh random task heads for baseline
baseline_regression_head = RegressionHead(encoded_dim=64, output_dim=1)
baseline_classification_head = ClassificationHead(encoded_dim=64, num_classes=5)
baseline_regression_head = baseline_regression_head.to(device)
baseline_classification_head = baseline_classification_head.to(device)

# Evaluate baseline
baseline_metrics = {}
baseline_regression_head.eval()
baseline_classification_head.eval()

with torch.no_grad():
    vol_preds = []
    vol_trues = []
    cong_preds = []
    cong_trues = []
    
    for batch in phase2_test_loader:
        X_tensor = torch.tensor(batch['X'], dtype=torch.float32).to(device)
        encoded = encoder_for_phase2(X_tensor)
        
        vol_pred = baseline_regression_head(encoded).squeeze().cpu().numpy()
        cong_logits = baseline_classification_head(encoded)
        cong_pred = cong_logits.argmax(dim=1).cpu().numpy()
        
        vol_preds.append(vol_pred)
        vol_trues.append(batch['y_volume'])
        cong_preds.append(cong_pred)
        cong_trues.append(batch['y_congestion'])

baseline_metrics['volume_preds'] = np.concatenate(vol_preds)
baseline_metrics['volume_trues'] = np.concatenate(vol_trues)
baseline_metrics['congestion_preds'] = np.concatenate(cong_preds)
baseline_metrics['congestion_trues'] = np.concatenate(cong_trues)

baseline_metrics['mae'] = np.mean(np.abs(baseline_metrics['volume_preds'] - baseline_metrics['volume_trues']))
baseline_metrics['rmse'] = np.sqrt(np.mean((baseline_metrics['volume_preds'] - baseline_metrics['volume_trues']) ** 2))
baseline_metrics['accuracy'] = np.mean(baseline_metrics['congestion_preds'] == baseline_metrics['congestion_trues'])
from sklearn.metrics import f1_score as sk_f1_score
baseline_metrics['macro_f1'] = sk_f1_score(baseline_metrics['congestion_trues'], 
                                           baseline_metrics['congestion_preds'], 
                                           average='macro', zero_division=0)

print(f"  Results:")
print(f"    MAE:  {baseline_metrics['mae']:.4f}")
print(f"    RMSE: {baseline_metrics['rmse']:.4f}")
print(f"    Accuracy: {baseline_metrics['accuracy']:.4f}")
print(f"    Macro-F1: {baseline_metrics['macro_f1']:.4f}")

print("\nModel B: Phase 2 Fine-tuned (This work)")
print("  - Phase 1 frozen encoder + fine-tuned task heads")
print("  - Represents transfer learning with downstream task optimization")
print(f"  Results:")
print(f"    MAE:  {phase2_test_metrics['mae']:.4f}")
print(f"    RMSE: {phase2_test_metrics['rmse']:.4f}")
print(f"    Accuracy: {phase2_test_metrics['accuracy']:.4f}")
print(f"    Macro-F1: {phase2_test_metrics['macro_f1']:.4f}")

print("\nImprovement (Phase 2 - Phase 1):")
print(f"  MAE:  {phase2_test_metrics['mae'] - baseline_metrics['mae']:.4f} " + 
      f"({100*(phase2_test_metrics['mae']-baseline_metrics['mae'])/baseline_metrics['mae']:.1f}%)")
print(f"  RMSE: {phase2_test_metrics['rmse'] - baseline_metrics['rmse']:.4f} " +
      f"({100*(phase2_test_metrics['rmse']-baseline_metrics['rmse'])/baseline_metrics['rmse']:.1f}%)")
print(f"  Accuracy: {phase2_test_metrics['accuracy'] - baseline_metrics['accuracy']:.4f} " +
      f"({100*(phase2_test_metrics['accuracy']-baseline_metrics['accuracy'])/baseline_metrics['accuracy']:.1f}%)")
print(f"  Macro-F1: {phase2_test_metrics['macro_f1'] - baseline_metrics['macro_f1']:.4f} " +
      f"({100*(phase2_test_metrics['macro_f1']-baseline_metrics['macro_f1'])/baseline_metrics['macro_f1']:.1f}%)")

print("="*80 + "\n")


Model A: Phase 1 Encoder Only (No fine-tuning)
  - Uses frozen encoder + random task heads
  - Represents baseline pre-trained features
  Results:
    MAE:  19.3736
    RMSE: 19.8703
    Accuracy: 0.2712
    Macro-F1: 0.0859

Model B: Phase 2 Fine-tuned (This work)
  - Phase 1 frozen encoder + fine-tuned task heads
  - Represents transfer learning with downstream task optimization
  Results:
    MAE:  9.5067
    RMSE: 10.5221
    Accuracy: 0.1822
    Macro-F1: 0.1549

Improvement (Phase 2 - Phase 1):
  MAE:  -9.8669 (-50.9%)
  RMSE: -9.3482 (-47.0%)
  Accuracy: -0.0890 (-32.8%)
  Macro-F1: 0.0690 (80.3%)



error taxonomy audit

In [26]:
# Error Analysis on Phase 2 Test Predictions

print("="*80)
print("ERROR ANALYSIS - PHASE 2 PREDICTIONS")
print("="*80)

# Classification error analysis
confusion_matrix = np.zeros((5, 5))
for true_label, pred_label in zip(phase2_test_metrics['congestion_trues'], 
                                   phase2_test_metrics['congestion_preds']):
    confusion_matrix[int(true_label), int(pred_label)] += 1

print("\nConfusion Matrix (rows=true, cols=predicted):")
print("     Pred 0  Pred 1  Pred 2  Pred 3  Pred 4")
for i in range(5):
    row_str = f"True {i}: "
    for j in range(5):
        row_str += f"{int(confusion_matrix[i,j]):6d}  "
    print(row_str)

# Identify problematic predictions
false_negatives = []  # Missed congestion (true >= 3, pred <= 1)
false_positives = []  # False alarms (true <= 1, pred >= 3)

for i, (true_cong, pred_cong) in enumerate(zip(phase2_test_metrics['congestion_trues'],
                                                phase2_test_metrics['congestion_preds'])):
    if true_cong >= 3 and pred_cong <= 1:
        false_negatives.append(i)
    elif true_cong <= 1 and pred_cong >= 3:
        false_positives.append(i)

print(f"\n\nError Breakdown:")
print(f"  False negatives (missed congestion): {len(false_negatives)}")
print(f"  False positives (false alarms): {len(false_positives)}")
print(f"  Correct: {np.sum(np.diag(confusion_matrix)):.0f}")
print(f"  Total test samples: {len(phase2_test_metrics['congestion_trues'])}")

# Regression error analysis
vol_errors = phase2_test_metrics['volume_preds'] - phase2_test_metrics['volume_trues']
print(f"\n\nRegression Error Distribution (Vehicle Count):")
print(f"  Mean absolute error: {np.mean(np.abs(vol_errors)):.4f}")
print(f"  Std dev: {np.std(np.abs(vol_errors)):.4f}")
print(f"  Min error: {vol_errors.min():.4f}")
print(f"  Max error: {vol_errors.max():.4f}")
print(f"  Median absolute error: {np.median(np.abs(vol_errors)):.4f}")

# Identify worst predictions
worst_indices = np.argsort(np.abs(vol_errors))[-5:]
print(f"\n  Top 5 worst regression predictions:")
for idx, i in enumerate(worst_indices):
    print(f"    {idx+1}. Predicted {phase2_test_metrics['volume_preds'][i]:.2f}, " +
          f"True {phase2_test_metrics['volume_trues'][i]:.2f}, " +
          f"Error {vol_errors[i]:.2f}")

print("="*80 + "\n")

ERROR ANALYSIS - PHASE 2 PREDICTIONS

Confusion Matrix (rows=true, cols=predicted):
     Pred 0  Pred 1  Pred 2  Pred 3  Pred 4
True 0:      7      25       8      13       0  
True 1:     11      21       8      25       0  
True 2:      6      13       6       5       0  
True 3:      4      17       7       9       0  
True 4:      6      29       6      10       0  


Error Breakdown:
  False negatives (missed congestion): 56
  False positives (false alarms): 38
  Correct: 43
  Total test samples: 236


Regression Error Distribution (Vehicle Count):
  Mean absolute error: 9.5067
  Std dev: 4.5097
  Min error: -23.6272
  Max error: 0.6222
  Median absolute error: 9.2000

  Top 5 worst regression predictions:
    1. Predicted 9.90, True 30.00, Error -20.10
    2. Predicted 9.71, True 30.00, Error -20.29
    3. Predicted 10.65, True 31.00, Error -20.35
    4. Predicted 10.23, True 31.00, Error -20.77
    5. Predicted 9.37, True 33.00, Error -23.63



saving final results

In [27]:
# Save Phase 2 Final Results

results_phase2 = {
    'phase1_pretrained': {
        'mae': float(phase2_test_metrics['mae']),
        'rmse': float(phase2_test_metrics['rmse']),
        'accuracy': float(phase2_test_metrics['accuracy']),
        'macro_f1': float(phase2_test_metrics['macro_f1'])
    },
    'baseline_random_heads': {
        'mae': float(baseline_metrics['mae']),
        'rmse': float(baseline_metrics['rmse']),
        'accuracy': float(baseline_metrics['accuracy']),
        'macro_f1': float(baseline_metrics['macro_f1'])
    },
    'improvement': {
        'mae_delta': float(phase2_test_metrics['mae'] - baseline_metrics['mae']),
        'rmse_delta': float(phase2_test_metrics['rmse'] - baseline_metrics['rmse']),
        'accuracy_delta': float(phase2_test_metrics['accuracy'] - baseline_metrics['accuracy']),
        'macro_f1_delta': float(phase2_test_metrics['macro_f1'] - baseline_metrics['macro_f1'])
    }
}

# Save results to JSON
with open('phase2_results.json', 'w') as f:
    json.dump(results_phase2, f, indent=2)

# Save model checkpoint for inference
model_checkpoint = {
    'encoder_state': encoder_for_phase2.state_dict(),
    'regression_head_state': regression_head.state_dict(),
    'classification_head_state': classification_head.state_dict(),
    'scaler_mean': scaler.mean.tolist(),
    'scaler_std': scaler.std.tolist(),
    'device': str(device)
}

torch.save(model_checkpoint, 'phase2_final_model.pt')

print("Saved Phase 2 results:")
print("  - phase2_results.json: Evaluation metrics and improvements")
print("  - phase2_final_model.pt: Complete trained model (encoder + heads)")
print("  - phase2_checkpoint.pt: Best checkpoint during training")
print("\nPhase 2 training complete!")

Saved Phase 2 results:
  - phase2_results.json: Evaluation metrics and improvements
  - phase2_final_model.pt: Complete trained model (encoder + heads)
  - phase2_checkpoint.pt: Best checkpoint during training

Phase 2 training complete!


comparative results table

In [28]:
# Final Results Summary Table

print("\n" + "="*100)
print("FINAL RESULTS - TEMPORAL CONVOLUTIONAL NETWORK (TCN) FOR TRAFFIC PREDICTION")
print("="*100)

results_table = [
    ['Model', 'MAE (vehicles)', 'RMSE (vehicles)', 'Accuracy (%)', 'Macro-F1'],
    ['─' * 25, '─' * 16, '─' * 18, '─' * 14, '─' * 12],
    ['Phase 1 Encoder Only', f"{baseline_metrics['mae']:.4f}", f"{baseline_metrics['rmse']:.4f}", 
     f"{100*baseline_metrics['accuracy']:.2f}", f"{baseline_metrics['macro_f1']:.4f}"],
    ['Phase 2 Fine-tuned', f"{phase2_test_metrics['mae']:.4f}", f"{phase2_test_metrics['rmse']:.4f}", 
     f"{100*phase2_test_metrics['accuracy']:.2f}", f"{phase2_test_metrics['macro_f1']:.4f}"],
    ['─' * 25, '─' * 16, '─' * 18, '─' * 14, '─' * 12],
    ['Improvement (Δ)', f"{phase2_test_metrics['mae']-baseline_metrics['mae']:.4f}", 
     f"{phase2_test_metrics['rmse']-baseline_metrics['rmse']:.4f}",
     f"{100*(phase2_test_metrics['accuracy']-baseline_metrics['accuracy']):.2f}",
     f"{phase2_test_metrics['macro_f1']-baseline_metrics['macro_f1']:.4f}"],
]

for row in results_table:
    print(f"{row[0]:<25} {row[1]:>16} {row[2]:>18} {row[3]:>14} {row[4]:>12}")

print("="*100)

print("\nTRAINING CONFIGURATION:")
print(f"  Phase 1 (Pre-training): Self-supervised reconstruction on 20% masked sequence positions")
print(f"  Phase 2 (Fine-tuning): Downstream tasks - Vehicle count (MAE) + Congestion level (CrossEntropy)")
print(f"  Encoder: Frozen (50,496 parameters)")
print(f"  Task heads: Trainable (4,358 parameters)")
print(f"\nTEST SET: {len(phase2_test_metrics['volume_trues'])} samples")
print("="*100 + "\n")


FINAL RESULTS - TEMPORAL CONVOLUTIONAL NETWORK (TCN) FOR TRAFFIC PREDICTION
Model                       MAE (vehicles)    RMSE (vehicles)   Accuracy (%)     Macro-F1
───────────────────────── ──────────────── ────────────────── ────────────── ────────────
Phase 1 Encoder Only               19.3736            19.8703          27.12       0.0859
Phase 2 Fine-tuned                  9.5067            10.5221          18.22       0.1549
───────────────────────── ──────────────── ────────────────── ────────────── ────────────
Improvement (Δ)                    -9.8669            -9.3482          -8.90       0.0690

TRAINING CONFIGURATION:
  Phase 1 (Pre-training): Self-supervised reconstruction on 20% masked sequence positions
  Phase 2 (Fine-tuning): Downstream tasks - Vehicle count (MAE) + Congestion level (CrossEntropy)
  Encoder: Frozen (50,496 parameters)
  Task heads: Trainable (4,358 parameters)

TEST SET: 236 samples

